# W08 — Assignment (SQL limpieza + Many-to-Many)

**Único entregable semanal.**

## Setup

In [8]:
from pathlib import Path
import duckdb
import os
#os.chdir("..")
print("cwd:", os.getcwd())

PROJECT_ROOT = Path(".").resolve()
RAW_CSV = PROJECT_ROOT / "data" / "raw" / "pscomppars.csv"
DB_PATH = PROJECT_ROOT / "data" / "exoplanets.duckdb"

if not RAW_CSV.exists():
    raise FileNotFoundError(f"Missing {RAW_CSV}. Run W01/W02 download first.")

def sql_path(p: Path) -> str:
    return "'" + p.resolve().as_posix().replace("'", "''") + "'"

con = duckdb.connect(str(DB_PATH))

con.execute("DROP VIEW IF EXISTS raw_ps")
con.execute(f"CREATE VIEW raw_ps AS SELECT * FROM read_csv_auto({sql_path(RAW_CSV)})")

con.sql("SELECT COUNT(*) AS n_raw FROM raw_ps").show()

cwd: c:\Users\nancy\Documents\Ingenieria_datos
┌───────┐
│ n_raw │
│ int64 │
├───────┤
│  6107 │
└───────┘



## Parte A — Limpieza Raw→Silver v2

In [9]:
con.sql("SELECT discoverymethod, COUNT(*) AS n FROM raw_ps WHERE discoverymethod IS NOT NULL GROUP BY discoverymethod ORDER BY n DESC LIMIT 15").show()
con.sql("SELECT COUNT(DISTINCT discoverymethod) AS n_unique_methods FROM raw_ps WHERE discoverymethod IS NOT NULL").show()

┌───────────────────────────────┬───────┐
│        discoverymethod        │   n   │
│            varchar            │ int64 │
├───────────────────────────────┼───────┤
│ Transit                       │  4501 │
│ Radial Velocity               │  1166 │
│ Microlensing                  │   266 │
│ Imaging                       │    92 │
│ Transit Timing Variations     │    39 │
│ Eclipse Timing Variations     │    17 │
│ Orbital Brightness Modulation │     9 │
│ Pulsar Timing                 │     8 │
│ Astrometry                    │     6 │
│ Pulsation Timing Variations   │     2 │
│ Disk Kinematics               │     1 │
├───────────────────────────────┴───────┤
│ 11 rows                     2 columns │
└───────────────────────────────────────┘

┌──────────────────┐
│ n_unique_methods │
│      int64       │
├──────────────────┤
│               11 │
└──────────────────┘



### TODO A1 — method_map

In [10]:
# TODO A1: crea una tabla method_map con mapeos raw -> canonical (>=6).

# TODO: CREATE TABLE + INSERTs
con.execute("DROP TABLE IF EXISTS method_map")

con.execute("CREATE TABLE method_map(raw_method VARCHAR, canonical_method VARCHAR)")

con.execute("""
INSERT INTO method_map VALUES
    ('Transit',                        'transit'),
    ('Radial Velocity',                'radial_velocity'),
    ('Imaging',                        'imaging'),
    ('Microlensing',                   'microlensing'),
    ('Transit Timing Variations',      'transit_timing_variations'),
    ('Eclipse Timing Variations',      'eclipse_timing_variations'),
    ('Astrometry',                     'astrometry'),
    ('Orbital Brightness Modulation',  'orbital_brightness_modulation'),
    ('Pulsar Timing',                  'pulsar_timing'),
    ('Pulsation Timing Variations',    'pulsation_timing_variations')
""")

con.sql("SELECT * FROM method_map").show()

con.sql("SELECT * FROM method_map").show()

┌───────────────────────────────┬───────────────────────────────┐
│          raw_method           │       canonical_method        │
│            varchar            │            varchar            │
├───────────────────────────────┼───────────────────────────────┤
│ Transit                       │ transit                       │
│ Radial Velocity               │ radial_velocity               │
│ Imaging                       │ imaging                       │
│ Microlensing                  │ microlensing                  │
│ Transit Timing Variations     │ transit_timing_variations     │
│ Eclipse Timing Variations     │ eclipse_timing_variations     │
│ Astrometry                    │ astrometry                    │
│ Orbital Brightness Modulation │ orbital_brightness_modulation │
│ Pulsar Timing                 │ pulsar_timing                 │
│ Pulsation Timing Variations   │ pulsation_timing_variations   │
├───────────────────────────────┴───────────────────────────────┤
│ 10 rows 

### TODO A2 — silver_planet_v2

In [11]:
# TODO A2: crea silver_planet_v2 con:
# - hostname_clean = LOWER(TRIM(hostname))
# - discoverymethod_clean = COALESCE(map, LOWER(TRIM(discoverymethod_norm)))
# - disc_era = CASE por década

con.execute("DROP TABLE IF EXISTS silver_planet_v2")

con.execute("""
CREATE TABLE silver_planet_v2 AS
SELECT
    pl_name,
    hostname,
    LOWER(TRIM(hostname))  AS hostname_clean,
    discoverymethod,
    LOWER(TRIM(discoverymethod)) AS discoverymethod_norm,
    COALESCE(
        m.canonical_method,
        LOWER(TRIM(discoverymethod))
    ) AS discoverymethod_clean,
    disc_year,
    CASE
        WHEN disc_year < 2000 THEN 'pre-2000'
        WHEN disc_year < 2010 THEN '2000s'
        WHEN disc_year < 2020 THEN '2010s'
        ELSE '2020s'
    END AS disc_era,
    pl_orbper,
    pl_rade,
    pl_bmasse,
    pl_eqt,
    sy_dist,
    ra,
    dec
FROM raw_ps
LEFT JOIN method_map m
    ON LOWER(TRIM(raw_ps.discoverymethod)) = LOWER(m.raw_method)
WHERE pl_name   IS NOT NULL
  AND hostname  IS NOT NULL
  AND (disc_year IS NULL OR disc_year BETWEEN 1980 AND 2026)
  AND (pl_rade   IS NULL OR (pl_rade > 0 AND pl_rade <= 30))
  AND (pl_bmasse IS NULL OR pl_bmasse > 0)
""")

con.sql("SELECT COUNT(*) AS n_rows FROM silver_planet_v2").show()
con.sql("SELECT COUNT(*) AS n_null_hosts FROM silver_planet_v2 WHERE hostname_clean IS NULL").show()
con.sql("SELECT discoverymethod_clean, COUNT(*) AS n FROM silver_planet_v2 WHERE discoverymethod_clean IS NOT NULL GROUP BY discoverymethod_clean ORDER BY n DESC LIMIT 15").show()

┌────────┐
│ n_rows │
│ int64  │
├────────┤
│   6101 │
└────────┘

┌──────────────┐
│ n_null_hosts │
│    int64     │
├──────────────┤
│            0 │
└──────────────┘

┌───────────────────────────────┬───────┐
│     discoverymethod_clean     │   n   │
│            varchar            │ int64 │
├───────────────────────────────┼───────┤
│ transit                       │  4500 │
│ radial_velocity               │  1166 │
│ microlensing                  │   266 │
│ imaging                       │    87 │
│ transit_timing_variations     │    39 │
│ eclipse_timing_variations     │    17 │
│ orbital_brightness_modulation │     9 │
│ pulsar_timing                 │     8 │
│ astrometry                    │     6 │
│ pulsation_timing_variations   │     2 │
│ disk kinematics               │     1 │
├───────────────────────────────┴───────┤
│ 11 rows                     2 columns │
└───────────────────────────────────────┘



## Parte B — Many-to-Many (toy schema)

In [12]:
# TODO B1: construye un ejemplo M:N (toy schema) y responde 2 preguntas.
#
# REQUISITO (para que cuente como completo):
# - Debes crear el esquema con **link table** + **PK/FK**:
#   - planet_demo(planet_id PRIMARY KEY, name NOT NULL)
#   - method_demo(method_id PRIMARY KEY, method_name UNIQUE NOT NULL)
#   - planet_method_demo(planet_id, method_id) con:
#       PRIMARY KEY (planet_id, method_id)
#       FOREIGN KEY (planet_id) REFERENCES planet_demo(planet_id)
#       FOREIGN KEY (method_id) REFERENCES method_demo(method_id)
#
# - Inserta al menos 4 planetas, 3 métodos y relaciones M:N (un planeta con 2 métodos).
#
# Nota idempotencia (FK-safe): dropea puente primero.
con.execute("DROP TABLE IF EXISTS planet_method_demo")
con.execute("DROP TABLE IF EXISTS method_demo")
con.execute("DROP TABLE IF EXISTS planet_demo")

# 1) Crea tablas (puedes copiar y adaptar este DDL)
# TODO: descomenta y ajusta si quieres
con.execute("""
    CREATE TABLE planet_demo(
        planet_id INTEGER PRIMARY KEY,
        name      VARCHAR NOT NULL
    )
""")
con.execute("""
    CREATE TABLE method_demo(
        method_id   INTEGER PRIMARY KEY,
        method_name VARCHAR NOT NULL UNIQUE
    )
""")
con.execute("""
    CREATE TABLE planet_method_demo(
        planet_id INTEGER NOT NULL,
        method_id INTEGER NOT NULL,
        PRIMARY KEY (planet_id, method_id),
        FOREIGN KEY (planet_id) REFERENCES planet_demo(planet_id),
        FOREIGN KEY (method_id) REFERENCES method_demo(method_id)
    )
""")

# 2) Inserta datos
# TODO: INSERTs en planet_demo, method_demo, planet_method_demo

con.execute("""
    INSERT INTO planet_demo VALUES
        (1, 'Kepler-22b'),
        (2, 'HD 209458 b'),
        (3, '51 Peg b'),
        (4, 'TRAPPIST-1b')
""")
con.execute("""
    INSERT INTO method_demo VALUES
        (10, 'transit'),
        (20, 'radial_velocity'),
        (30, 'imaging')
""")
con.execute("""
    INSERT INTO planet_method_demo VALUES
        (1, 10),   -- Kepler-22b: transit
        (1, 20),   -- Kepler-22b: radial_velocity (M:N: 1 planeta, 2 métodos)
        (2, 10),   -- HD 209458 b: transit
        (2, 20),   -- HD 209458 b: radial_velocity
        (3, 20),   -- 51 Peg b: radial_velocity
        (4, 10)    -- TRAPPIST-1b: transit
""")

# Q1: # planetas por método (usa COUNT(DISTINCT planet_id))
q1 = """
SELECT m.method_name, COUNT(DISTINCT pm.planet_id) AS n_planets
FROM planet_method_demo pm
JOIN method_demo m ON pm.method_id = m.method_id
GROUP BY m.method_name
ORDER BY n_planets DESC
"""
con.sql(q1).show()

# Q2: # métodos por planeta
q2 = """
SELECT p.name, COUNT(DISTINCT pm.method_id) AS n_methods
FROM planet_method_demo pm
JOIN planet_demo p ON pm.planet_id = p.planet_id
GROUP BY p.name
ORDER BY n_methods DESC
"""
con.sql(q2).show()

┌─────────────────┬───────────┐
│   method_name   │ n_planets │
│     varchar     │   int64   │
├─────────────────┼───────────┤
│ transit         │         3 │
│ radial_velocity │         3 │
└─────────────────┴───────────┘

┌─────────────┬───────────┐
│    name     │ n_methods │
│   varchar   │   int64   │
├─────────────┼───────────┤
│ Kepler-22b  │         2 │
│ HD 209458 b │         2 │
│ TRAPPIST-1b │         1 │
│ 51 Peg b    │         1 │
└─────────────┴───────────┘



In [13]:
# TODO B2 (REQUERIDO): check de duplicados en la link table (debe dar 0 filas)
# Si tu PK compuesta está bien, este check debería retornar vacío.

con.sql("""
SELECT planet_id, method_id, COUNT(*) AS c
FROM planet_method_demo
GROUP BY planet_id, method_id
HAVING COUNT(*) > 1
""").show()

# (Opcional) intenta insertar un duplicado para ver que la PK compuesta lo bloquea
# try:
#     con.execute("INSERT INTO planet_method_demo VALUES (1, 10)")
# except Exception as e:
#     print("OK (PK compuesta bloquea duplicado):", str(e).splitlines()[0])

┌───────────┬───────────┬───────┐
│ planet_id │ method_id │   c   │
│   int32   │   int32   │ int64 │
├───────────┴───────────┴───────┤
│            0 rows             │
└───────────────────────────────┘



## Entregable único semanal (W08)
- Ejecuta el assignment.
- Entrega:
  1) `docs/w08_report.md` (copiar template)
  2) 1 entrada nueva en `docs/decisions_log.md` (copiar template)

**Extra requerido (M:N):** incluye evidencia de PK/FK en tu DDL y/o el check `HAVING COUNT(*)>1` retornando vacío.